# Review data in app.sqlite

This notebook loads the local SQLite database and shows users, reviews, entities, and review-entity links.

In [10]:
import pandas as pd
import sqlite3
from pathlib import Path

db_path = Path('../entity-driven/data/app.sqlite')
if not db_path.exists():
    raise FileNotFoundError(f'Database not found at {db_path.resolve()}')

conn = sqlite3.connect(db_path)
conn.row_factory = sqlite3.Row

def show_old(query, params=None):
    params = params or ()
    cur = conn.execute(query, params)
    rows = cur.fetchall()
    return [dict(row) for row in rows]


def show(query, params=None):
    params = params or ()
    cur = conn.execute(query, params)
    rows = cur.fetchall()
    return pd.DataFrame(rows, columns=rows[0].keys() if rows else [])


In [11]:
# Users
show('SELECT * FROM user ORDER BY id')

,id,user_id,password
0,1,lbk,1234
1,2,kbl,1234


In [12]:
# Reviews (latest first)
show(
    'SELECT id, user_id, created_at, updated_at, content FROM review ORDER BY COALESCE(updated_at, created_at) DESC'
)

,id,user_id,created_at,updated_at,content
0,22,1,2025-12-29T09:21:47.577Z,2025-12-29T09:21:47.577Z,앱에서 맥주 재고 확인하고 직접 마트 가니까 아무리 찾아도 못찾겠다. 집에 와서 보...
1,21,2,2025-12-29T09:20:45.197Z,2025-12-29T09:20:45.197Z,4개 한셋트 치약 1+1 세일한다기에 사고 보니 3개 한셋트 치약이었다. 영수증엔 ...
2,20,2,2025-12-29T09:20:17.511Z,2025-12-29T09:20:17.511Z,홈플러스 앱으로 와인 주문: 10만원 이상 사면 20%할인한다기에 가격 맞춰 3병 ...
3,19,1,2025-12-29T09:19:50.904Z,2025-12-29T09:19:50.904Z,정자동 정동마트에서 종종 바나나 세일을 하는데 대부분 거무죽죽하니 신선도 관리를 어...
4,18,1,2025-12-29T09:18:58.976Z,2025-12-29T09:18:58.976Z,예전에 땡겨요에서 세일하길래 무슨 요리 배달 주문했더니 배달에 한시간 이상 걸린다고...
5,17,2,2025-12-29T09:18:02.078Z,2025-12-29T09:18:02.078Z,며칠이 니자도 발뒤꿈치가 아파 미금역 근처 정형외과를 찾아갔다. 아킬레스건염 같은 ...
6,16,2,2025-12-29T09:17:13.163Z,2025-12-29T09:17:13.163Z,발뒤꿈치가 아파 정형외과를 찾는데 저번에 비싼 주사 맞은데를 피해서 다른 병원을 갔...
7,15,2,2025-12-29T09:16:18.396Z,2025-12-29T09:16:18.396Z,엄지 발가락이 아파서 동네 정형외과 갔다가 얼떨결에 비급여 10만원짜리 주사 맞고 ...
8,14,1,2025-12-29T09:15:40.337Z,2025-12-29T09:15:40.337Z,속초에서 시장 구경하고 돌아가는 길 닭강정 사는걸 깜빡했다. 많은 사람들이 만석닭강...
9,13,1,2025-12-29T09:15:16.784Z,2025-12-29T09:15:16.784Z,마트에서 풀무원 감자전/오징어부추전 구입. 에어프라이로 조리했는데 기름 범벅. 후라...


In [13]:
# Entities
show('SELECT * FROM entity ORDER BY id')

,id,name,type,level,parent_id
0,1,BHC 쏘마치,None,None,None
1,2,처가집 양념통닭 K마라치킨,None,None,None
2,3,멕시카나,None,None,None
3,4,배,None,None,None
4,5,비비큐 맵소,None,None,None
5,6,노랑통닭,None,None,None
6,9,배민,None,None,None
7,10,비비큐 맵소디,None,None,None
8,11,푸라닭 깐풍치킨,None,None,None
9,12,교촌치킨 간장/레드 반반,None,None,None


In [14]:
# Review-entity links
show(
    'SELECT review_id, entity_id FROM review_entity ORDER BY review_id, entity_id'
)

,review_id,entity_id
0,1,1
1,2,2
2,3,3
3,3,9
4,4,10
5,5,6
6,6,11
7,7,12
8,8,13
9,8,14


In [7]:
# Joined view: reviews with entity names
show(
    '''
    SELECT r.id, r.user_id, r.created_at, r.updated_at, r.content,
           GROUP_CONCAT(e.name, ', ') AS entities
    FROM review r
    LEFT JOIN review_entity re ON r.id = re.review_id
    LEFT JOIN entity e ON e.id = re.entity_id
    GROUP BY r.id
    ORDER BY COALESCE(r.updated_at, r.created_at) DESC
    '''
)

[{'id': 22,
  'user_id': 1,
  'created_at': '2025-12-29T09:21:47.577Z',
  'updated_at': '2025-12-29T09:21:47.577Z',
  'content': '앱에서 맥주 재고 확인하고 직접 마트 가니까 아무리 찾아도 못찾겠다. 집에 와서 보니 여전히 앱에는 재고가 있다. 어디에 있었던걸까?',
  'entities': '이마트'},
 {'id': 21,
  'user_id': 2,
  'created_at': '2025-12-29T09:20:45.197Z',
  'updated_at': '2025-12-29T09:20:45.197Z',
  'content': '4개 한셋트 치약 1+1 세일한다기에 사고 보니 3개 한셋트 치약이었다. 영수증엔 여전히 4개 셋트로 표시되어있다. 귀찮아서 환불안하고 그냥 쓰려한다',
  'entities': '이마트'},
 {'id': 20,
  'user_id': 2,
  'created_at': '2025-12-29T09:20:17.511Z',
  'updated_at': '2025-12-29T09:20:17.511Z',
  'content': '홈플러스 앱으로 와인 주문: 10만원 이상 사면 20%할인한다기에 가격 맞춰 3병 결재하고 며칠뒤 배송을 기다리는데 배송 당일 2병 재고가 없다고 2병 취소 시키고 하나만 받으러 오라고 한다. 그러면 할인이 안되는데 살 이유가 없다. 다 취소하고 얼마뒤 보니 재고 없다던 와인 또 팔고 있더라. 또 주문했다가 당할까봐 그냥 포기',
  'entities': '홈플러스'},
 {'id': 19,
  'user_id': 1,
  'created_at': '2025-12-29T09:19:50.904Z',
  'updated_at': '2025-12-29T09:19:50.904Z',
  'content': '정자동 정동마트에서 종종 바나나 세일을 하는데 대부분 거무죽죽하니 신선도 관리를 어찌 하는지 황당하다',
  'en

In [ ]:
conn.close()